# EDA — Panel de deforestación de Colombia

Análisis exploratorio de **uno** de los dos paneles finales que produce el
pipeline (nunca se mezclan — ver METODOLOGIA.md decisión 15):

- `panel_deforestacion_colombia_dist_alert.csv`/`.parquet` (2023-presente,
  `python main_local.py consolidar --fuente dist_alert`)
- `panel_deforestacion_colombia_gfw.csv`/`.parquet` (2020-presente,
  `python main_local.py consolidar --fuente gfw`)

Este notebook carga uno a la vez, seleccionado en la celda de código de la
sección 1 (`FUENTE = "dist_alert"` o `"gfw"`).

**Antes de leer este notebook**: la definición de cada variable, las fuentes
de datos y los supuestos metodológicos están en
[`METODOLOGIA.md`](METODOLOGIA.md); el detalle de cómo se generó cada
archivo está en [`GUIA_CODIGO.md`](GUIA_CODIGO.md). Este notebook asume esa
metodología como dada y se concentra en **describir los datos**, no en
justificarlos ni en modelarlos.

**Qué es cada celda-mes, en una frase**: una celda de 5×5 km de Colombia, en
un mes dado, con cuántas hectáreas de bosque perdió ese mes (`area_def_ha`)
y cuánto bosque le quedaba disponible al empezar ese mes
(`bosque_remanente_ha`).

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
from matplotlib.colors import LinearSegmentedColormap

from mapa_folium import cargar_panel   # reutiliza el mismo loader robusto (parquet -> csv) del resto del pipeline

# ---------------------------------------------------------------------
# Paleta y estilo — valores tomados tal cual de la paleta validada del
# proyecto (secuencial azul para magnitud, par divergente azul<->rojo
# para polaridad/correlacion, naranja como acento categorico de "evento").
# No se inventan colores nuevos: son los mismos hex ya validados.
# ---------------------------------------------------------------------
INK, INK_2, MUTED = "#0b0b0b", "#52514e", "#898781"
GRID, BASE, SURFACE = "#e1e0d9", "#c3c2b7", "#fcfcfb"
AZUL, NARANJA = "#256abf", "#eb6834"

SEQ_STEPS = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]
cmap_seq = LinearSegmentedColormap.from_list("seq_azul", SEQ_STEPS)
cmap_div = LinearSegmentedColormap.from_list("div_azul_rojo", ["#e34948", "#f0efec", "#256abf"])

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": BASE, "axes.labelcolor": INK_2, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "grid.color": GRID,
    "axes.grid": True, "grid.linewidth": 0.6, "grid.alpha": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 11, "figure.dpi": 110,
})

MESES_ES = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]


def miles(ax, eje="y"):
    # Formatea los ticks de un eje con separador de miles.
    fmt = mticker.FuncFormatter(lambda x, _: f"{x:,.0f}")
    (ax.yaxis if eje == "y" else ax.xaxis).set_major_formatter(fmt)

## 1. Carga de datos

`FUENTE` selecciona cuál de los dos paneles independientes se analiza en
todo el resto del notebook (ver introducción arriba).

In [ ]:
FUENTE = "dist_alert"   # o "gfw" -- los dos paneles nunca se mezclan (METODOLOGIA.md decision 15)

df = cargar_panel(FUENTE)
print(f"Fuente: {FUENTE}")
print(f"Filas: {len(df):,}  |  Columnas: {df.shape[1]}")
df.head()

## 2. Estructura general

In [ ]:
df.info()

In [ ]:
faltantes = df.isna().sum()
faltantes = faltantes[faltantes > 0].sort_values(ascending=False)
print("Columnas con datos faltantes (esperado: los rezagos y la media movil,")
print("que no existen para los primeros meses de cada celda):")
faltantes

In [ ]:
df.describe().T

## 3. Balance del panel

`consolidar.py` garantiza que **toda** celda tenga una fila en **todo**
periodo (`balancear()`, ver METODOLOGIA.md §3 y GUIA_CODIGO.md §7). Se
verifica aquí: filas debe ser exactamente celdas × periodos.

In [ ]:
n_celdas = df["cell_id"].nunique()
n_periodos = df["periodo"].nunique()
n_deptos = df["departamento"].nunique()

print(f"Celdas únicas   : {n_celdas:,}")
print(f"Periodos (meses): {n_periodos}  ({df['periodo'].min().date()} a {df['periodo'].max().date()})")
print(f"Departamentos   : {n_deptos}")
print(f"Filas esperadas (celdas x periodos): {n_celdas * n_periodos:,}")
print(f"Filas reales                       : {len(df):,}")
assert n_celdas * n_periodos == len(df), "El panel NO esta balanceado -- revisar consolidar.py"
print("\nEl panel es un rectangulo perfecto: sin huecos.")

## 4. Desbalance de clases — la variable `evento`

`evento` es el target binario del pipeline (1 si hubo algo de
deforestación confirmada ese mes en esa celda). Antes de cualquier análisis
posterior hay que ver qué tan raro es el evento — determina, por ejemplo,
que un modelo posterior necesite tratarlo como un problema de clasificación
muy desbalanceada (de ahí el diseño de dos etapas tipo *hurdle*, ver
METODOLOGIA.md §1.1 y §6).

In [ ]:
conteo = df["evento"].value_counts().sort_index()
pct = conteo / conteo.sum() * 100

fig, ax = plt.subplots(figsize=(6, 4.2))
bars = ax.bar(["Sin evento (0)", "Con evento (1)"], conteo.values, color=[AZUL, NARANJA], width=0.55)
for b, v, p in zip(bars, conteo.values, pct.values):
    ax.text(b.get_x() + b.get_width() / 2, v, f"{v:,.0f}\n({p:.2f}%)",
            ha="center", va="bottom", fontsize=10, color=INK)
ax.set_ylabel("Celda-mes")
ax.set_title("Desbalance de clases: la variable evento")
ax.margins(y=0.18)
miles(ax)
plt.tight_layout()
plt.show()

print(f"Solo el {pct.loc[1]:.2f}% de las celda-mes registraron un evento de deforestacion confirmada.")

## 5. Evolución temporal nacional

Suma de `area_def_ha` de todas las celdas, por mes — la serie de tiempo más
agregada posible del fenómeno.

**Nota de lectura**: por el diseño de DIST-ALERT (ver METODOLOGIA.md §2.2),
los **últimos 1-2 meses** de cualquier corrida están probablemente
subestimados: parte de sus eventos reales todavía no han pasado a estado
*confirmado* al momento de la descarga. No interprete una caída en el tramo
final de la serie como una mejora real sin revisar la fecha de la última
descarga.

In [ ]:
serie = df.groupby("periodo")["area_def_ha"].sum().sort_index()

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(serie.index, serie.values, color=AZUL, linewidth=2)
ax.fill_between(serie.index, serie.values, color=AZUL, alpha=0.12)

pico = serie.idxmax()
ax.scatter([pico], [serie.max()], color=NARANJA, zorder=5, s=45)
ax.annotate(f"{serie.max():,.0f} ha\n{pico.strftime('%Y-%m')}", xy=(pico, serie.max()),
            xytext=(10, 8), textcoords="offset points", fontsize=9, color=INK_2)

ax.set_ylabel("Hectáreas deforestadas (nacional)")
ax.set_title("Deforestación confirmada por mes — Colombia")
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
miles(ax)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

### ¿Por qué el primer periodo se ve tan alto?

El primer periodo del panel no mide "deforestación nueva ocurrida ese mes":
mide todo lo que DIST-ALERT ya tenía en estado *confirmado* en la primera
instantánea que este pipeline descargó para cada tile, sin importar cuánto
llevaba acumulándose antes. La máscara `ya_contado` (ver METODOLOGIA.md §3.4)
arranca vacía: todo píxel que ya estaba confirmado en esa primera instantánea
entra de una sola vez, así su `VEG-DIST-DATE` real caiga justo en ese primer
mes. En la práctica, el primer periodo mide un **stock inicial** acumulado
antes del arranque del monitoreo, no un **flujo mensual** comparable al resto
de la serie — y por eso conviene excluirlo de promedios y comparaciones entre
meses. Las secciones siguientes (estacionalidad, resumen ejecutivo) ya lo
hacen así explícitamente.

In [ ]:
primero = serie.iloc[0]
resto = serie.iloc[1:]
razon = primero / resto.median()

print(f"Periodo 1 ({serie.index[0].strftime('%Y-%m')})        : {primero:,.0f} ha")
print(f"Mediana de los periodos 2..{len(serie)} : {resto.median():,.0f} ha")
print(f"El primer periodo es {razon:.1f}x la mediana del resto de la serie.")

In [ ]:
acumulado = serie.cumsum()

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(acumulado.index, acumulado.values, color=AZUL, linewidth=2)
ax.fill_between(acumulado.index, acumulado.values, color=AZUL, alpha=0.10)
ax.annotate(f"{acumulado.iloc[-1]:,.0f} ha totales", xy=(acumulado.index[-1], acumulado.iloc[-1]),
            xytext=(-100, 8), textcoords="offset points", fontsize=10, color=INK, fontweight="bold")
ax.set_ylabel("Hectáreas acumuladas")
ax.set_title("Deforestación acumulada, nacional")
miles(ax)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 6. Estacionalidad

Promedio de deforestación nacional por mes calendario, a través de todos los
años que cubre el panel. **Se excluye el primer periodo** (ver sección 5):
incluirlo sesgaría fuertemente hacia arriba el promedio de su mes calendario,
mezclando un artefacto de inicialización con estacionalidad real.

**Cuidado adicional**: no todos los meses tienen el mismo número de años
detrás. Si la ventana temporal empieza en enero y termina a mitad de año, los
meses de la segunda mitad del año tienen un año menos de datos que los de la
primera mitad — la celda de código lo verifica explícitamente antes de
graficar, para no comparar promedios calculados sobre bases distintas sin
decirlo.

In [ ]:
# Se excluye el periodo 1 (indice 0): es el stock inicial, no un flujo
# mensual comparable (ver seccion 5). serie ya viene ordenada por fecha.
serie_df = serie.iloc[1:].reset_index()
serie_df["mes"] = serie_df["periodo"].dt.month
serie_df["anio"] = serie_df["periodo"].dt.year

n_anios_por_mes = serie_df.groupby("mes")["anio"].nunique()
if n_anios_por_mes.nunique() > 1:
    print("Aviso: no todos los meses tienen el mismo numero de anios de historia:")
    print(n_anios_por_mes.to_string())
    print()

promedio_mes = serie_df.groupby("mes")["area_def_ha"].mean()

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(promedio_mes.index, promedio_mes.values, color=AZUL, width=0.62)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(MESES_ES)
ax.set_ylabel("Hectáreas promedio por mes (nacional)")
ax.set_title(f"Estacionalidad promedio de la deforestación ({serie_df['anio'].min()}–{serie_df['anio'].max()}, sin periodo 1)")
miles(ax)
plt.tight_layout()
plt.show()

top3_meses = promedio_mes.sort_values(ascending=False).head(3)
print("Meses con mayor deforestación promedio (excluyendo el periodo 1):")
for m, v in top3_meses.items():
    print(f"  {MESES_ES[m-1]:4s}: {v:,.0f} ha (n={n_anios_por_mes[m]} años)")

## 7. Distribución de la tasa de deforestación (`tasa_def`)

Condicionada a que hubo evento (el caso `evento=0` es, por definición,
`tasa_def=0` — ver la sección 4). Escala logarítmica en el eje y porque la
distribución es muy asimétrica: la mayoría de eventos son pérdidas pequeñas
relativas al bosque remanente de la celda.

In [ ]:
con_evento = df[df["evento"] == 1]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(con_evento["tasa_def"], bins=60, color=AZUL, edgecolor=SURFACE, linewidth=0.3)
ax.set_yscale("log")
ax.set_xlabel("tasa_def  (fracción del bosque remanente deforestada ese mes)")
ax.set_ylabel("Celda-mes (escala log)")
ax.set_title("Distribución de la tasa de deforestación, dado que hubo evento")
plt.tight_layout()
plt.show()

con_evento["tasa_def"].describe()

## 8. Ranking de departamentos

In [ ]:
por_depto = df.groupby("departamento")["area_def_ha"].sum().sort_values(ascending=False)
top15 = por_depto.head(15)

norm = (top15.values - top15.values.min()) / (top15.values.max() - top15.values.min() + 1e-9)
colores = cmap_seq(0.28 + 0.7 * norm)

fig, ax = plt.subplots(figsize=(9, 7))
y_pos = np.arange(len(top15))
ax.barh(y_pos, top15.values, color=colores)
ax.set_yticks(y_pos)
ax.set_yticklabels(top15.index)
ax.invert_yaxis()
for i, v in enumerate(top15.values):
    ax.text(v, i, f" {v:,.0f} ha", va="center", fontsize=9, color=INK_2)
ax.set_xlabel("Hectáreas deforestadas (total del panel)")
ax.set_title("Los 15 departamentos con más deforestación confirmada")
miles(ax, "x")
plt.tight_layout()
plt.show()

top1_share = por_depto.iloc[0] / por_depto.sum() * 100
top3_share = por_depto.iloc[:3].sum() / por_depto.sum() * 100
print(f"{por_depto.index[0]} concentra el {top1_share:.1f}% del total nacional.")
print(f"Los 3 departamentos con más pérdida concentran el {top3_share:.1f}% del total.")

## 9. Concentración espacial — ¿pocas celdas explican la mayoría de la pérdida?

Curva de concentración (estilo Lorenz): celdas ordenadas de mayor a menor
deforestación total, contra qué porcentaje acumulado de la pérdida nacional
explican. Si la deforestación estuviera repartida uniformemente entre todas
las celdas, la curva coincidiría con la diagonal punteada.

In [ ]:
por_celda = df.groupby("cell_id")["area_def_ha"].sum().sort_values(ascending=False)
total = por_celda.sum()
cum_share = por_celda.cumsum() / total
celda_share = np.arange(1, len(por_celda) + 1) / len(por_celda) * 100

fig, ax = plt.subplots(figsize=(7, 6.5))
ax.plot(celda_share, cum_share.values * 100, color=AZUL, linewidth=2, label="Curva real")
ax.plot([0, 100], [0, 100], color=MUTED, linewidth=1, linestyle="--", label="Reparto uniforme")

marcas = {}
for p in (5, 10, 20):
    idx = int(len(por_celda) * p / 100) - 1
    y = cum_share.iloc[idx] * 100
    marcas[p] = y
    ax.scatter([p], [y], color=NARANJA, zorder=5, s=35)
    ax.annotate(f"{p}% celdas → {y:.0f}% de la pérdida", xy=(p, y),
                xytext=(8, -4), textcoords="offset points", fontsize=8.5, color=INK_2)

ax.set_xlabel("% de celdas (ordenadas de mayor a menor pérdida)")
ax.set_ylabel("% acumulado de la deforestación total")
ax.set_title("Concentración espacial de la deforestación")
ax.legend(frameon=False, fontsize=9, loc="lower right")
plt.tight_layout()
plt.show()

## 10. Mapa espacial (vista estática)

Cada punto es una celda con algo de deforestación acumulada en la ventana
completa; el color codifica la magnitud (escala logarítmica, porque la
distribución está muy concentrada — ver sección 9). Para explorar el mapa de
forma interactiva, con clic para ver el detalle de cada celda, use
`python mapa_folium.py` (genera `datos/panel/mapa_deforestacion.html`).

In [ ]:
por_celda_geo = (df.groupby("cell_id")
                  .agg(lon=("lon", "first"), lat=("lat", "first"), total_ha=("area_def_ha", "sum"))
                  .reset_index())
con_perdida = por_celda_geo[por_celda_geo["total_ha"] > 0]

fig, ax = plt.subplots(figsize=(7, 9.5))
sc = ax.scatter(con_perdida["lon"], con_perdida["lat"], c=np.log1p(con_perdida["total_ha"]),
                 cmap=cmap_seq, s=6, alpha=0.8, linewidths=0)
ax.set_aspect("equal")
ax.set_xlabel("Longitud")
ax.set_ylabel("Latitud")
ax.set_title("Deforestación acumulada por celda")
cbar = fig.colorbar(sc, ax=ax, shrink=0.55, label="log(1 + ha totales)")
plt.tight_layout()
plt.show()

## 11. Bosque remanente vs. tasa de deforestación

¿Las celdas con menos bosque remanente se deforestan proporcionalmente más
rápido, o más lento? `hexbin` en vez de un scatter simple porque con más de
un millón de puntos un scatter se satura visualmente (sobre-trazado) — el
color aquí es un conteo de celda-mes, no una magnitud de deforestación.

In [ ]:
sub = con_evento[con_evento["bosque_remanente_ha"] > 0]

fig, ax = plt.subplots(figsize=(7.5, 5.8))
# bins="log": el conteo por hexagono tambien esta muy concentrado (unos
# pocos hexagonos con miles de celda-mes vs. la mayoria con pocas), asi
# que sin esto la escala de color queda estirada por esos pocos hexagonos
# y el resto del grafico se ve plano/palido. Con bins="log" el color
# codifica log10(conteo), y el patron general vuelve a ser legible.
hb = ax.hexbin(sub["bosque_remanente_ha"], sub["tasa_def"], gridsize=40, cmap=cmap_seq,
               xscale="log", yscale="log", mincnt=1, bins="log")
ax.set_xlabel("Bosque remanente al inicio del periodo (ha, escala log)")
ax.set_ylabel("tasa_def (escala log)")
ax.set_title("Bosque remanente vs. tasa de deforestación (celdas con evento)")
fig.colorbar(hb, ax=ax, label="celda-mes (conteo, escala log)")
plt.tight_layout()
plt.show()

corr = np.corrcoef(np.log1p(sub["bosque_remanente_ha"]), np.log1p(sub["tasa_def"]))[0, 1]
print(f"Correlación (log-log) entre bosque remanente y tasa_def: {corr:.3f}")

## 12. Autocorrelación temporal — ¿la deforestación pasada predice la futura?

Correlación de Pearson entre `area_def_ha` y sus propios rezagos
(`lag_1_ha`, `lag_2_ha`, `lag_3_ha`) y la media móvil de 3 meses, calculada
sobre el panel completo (incluye los ceros — esto es la autocorrelación
"cruda" de la serie, no solo entre eventos). Escala divergente (azul↔rojo,
gris = sin correlación) porque el signo es lo que importa aquí, no solo la
magnitud.

In [ ]:
cols_lag = ["area_def_ha", "lag_1_ha", "lag_2_ha", "lag_3_ha", "media_movil_3"]
corr_mat = df[cols_lag].corr()

fig, ax = plt.subplots(figsize=(6, 5.5))
im = ax.imshow(corr_mat.values, cmap=cmap_div, vmin=-1, vmax=1)
ax.set_xticks(range(len(cols_lag)))
ax.set_xticklabels(cols_lag, rotation=40, ha="right")
ax.set_yticks(range(len(cols_lag)))
ax.set_yticklabels(cols_lag)
for i in range(len(cols_lag)):
    for j in range(len(cols_lag)):
        val = corr_mat.values[i, j]
        ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                color="white" if abs(val) > 0.55 else INK, fontsize=9)
fig.colorbar(im, ax=ax, label="correlación de Pearson", shrink=0.85)
ax.set_title("Autocorrelación: deforestación actual vs. rezagos")
plt.tight_layout()
plt.show()

## 13. Resumen ejecutivo de la EDA

Los números siguientes se calculan a partir de las celdas anteriores (nada
está copiado a mano), así que quedan actualizados automáticamente si se
vuelve a correr este notebook contra un panel más reciente.

In [ ]:
total_ha = df["area_def_ha"].sum()
pct_evento = df["evento"].mean() * 100
lag1_corr = corr_mat.loc["area_def_ha", "lag_1_ha"]
idx5 = int(len(por_celda) * 0.05) - 1
share5 = cum_share.iloc[idx5] * 100
mes_pico_real = resto.idxmax()   # excluye el periodo 1 (stock inicial, seccion 5)

print("RESUMEN EJECUTIVO")
print("=" * 60)
print(f"Ventana analizada          : {df['periodo'].min().date()} a {df['periodo'].max().date()} ({n_periodos} meses)")
print(f"Celdas / departamentos     : {n_celdas:,} / {n_deptos}")
print(f"Deforestación total        : {total_ha:,.0f} ha")
print(f"% de celda-mes con evento  : {pct_evento:.2f}%")
print(f"Depto. con más pérdida     : {por_depto.index[0]} ({top1_share:.1f}% del total nacional)")
print(f"Periodo 1 ({serie.index[0].strftime('%Y-%m')})           : {primero:,.0f} ha -- stock inicial, no comparable (ver sección 5)")
print(f"Mes de mayor deforestación : {mes_pico_real.strftime('%Y-%m')} ({resto.max():,.0f} ha)  [excluyendo el periodo 1]")
print(f"Concentración espacial     : el 5% de las celdas con más pérdida explica el {share5:.0f}% del total")
print(f"Correlación area_def_ha vs lag_1_ha: {lag1_corr:.3f}")

---

**Siguientes pasos sugeridos** (fuera del alcance de este notebook, que es
solo exploratorio):

1. **Excluir el primer periodo del panel** (sección 5) antes de calcular
   cualquier tasa promedio, estacionalidad o entrenar un modelo — es un
   artefacto de inicialización (stock acumulado antes de empezar a
   monitorear), no una observación mensual comparable al resto.
2. Dado el fuerte desbalance de clases y la concentración espacial
   documentados arriba, un modelo predictivo probablemente deba tratarse
   como un problema de dos etapas (clasificación `evento` + magnitud
   `tasa_def | evento=1`, ver METODOLOGIA.md §1.1).
3. Conviene revisar si hace falta ponderar o estratificar por departamento,
   dado lo concentrado del fenómeno (secciones 8-9).